# 20. Final Trust Labels (`src/models/trust_label_rules.csv` Audit & Breakdown)

**Topic**: Full Step 6.1d audit of standardized CSV-driven trust-label rules (`valid`, `invalid`, `excluded`), conflict detection, empty `closed_date` gating vs `unclosed` components, sensitivity analyses (`S1`–`S3`), chronological 80/20 split balance, and `road` / `noise` temporal diagnostics.

- **Inputs**:
  - `data/processed/complaints_clean.csv` (236,507 Brooklyn complaints, 2025-11 to 2026-10)
  - `data/processed/resolution_lookup.csv` (152 normalized 311 resolution descriptions)
  - `src/models/trust_label_rules.csv` (37 standardized single-label substring rules)
- **Outputs**:
  - Verified tables **(a)**–**(g)** and Step 6.1d integrity checks (read-only audit; no files written)
- **Runtime**: < 10 seconds

In [1]:
import sys; from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists()); sys.path.insert(0, str(ROOT))
import warnings
warnings.filterwarnings("ignore")
import hashlib
import pandas as pd
import numpy as np

from src.nbutils import Checker
from src.models.trust import (
    TRUST_RULES_PATH,
    load_trust_label_rules,
    classify_resolution_text,
    assign_trust_labels,
)

pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 120)

# Configuration Flags
RUN_HEAVY = False
RUN_API = False


In [2]:
# Section (a): Load Inputs, Rule Conflict Validation, and All 111 Distinct Resolved Texts
complaints_df = pd.read_csv(
    ROOT / "data/processed/complaints_clean.csv",
    dtype={"id": str, "resolution_id": str},
    keep_default_na=False,
)
lookup_df = pd.read_csv(
    ROOT / "data/processed/resolution_lookup.csv",
    dtype={"resolution_id": str},
    keep_default_na=False,
)
rules = load_trust_label_rules()
rules_rel = TRUST_RULES_PATH.relative_to(ROOT).as_posix()

complaints_df["ts"] = pd.to_datetime(complaints_df["timestamp"])
complaints_df["month_created"] = complaints_df["ts"].dt.to_period("M").astype(str)

# Gate vs full unclosed definition (required in code to explain 4,414 vs 4,175)
empty_closed_mask = complaints_df["closed_date"].str.strip() == ""
non_closed_status_mask = complaints_df["status"].str.strip() != "Closed"
unclosed_mask = empty_closed_mask | non_closed_status_mask
unclosed = unclosed_mask.astype(int)
complaints_df["empty_closed"] = empty_closed_mask.astype(int)
complaints_df["non_closed_status"] = non_closed_status_mask.astype(int)
complaints_df["unclosed"] = unclosed

resolved_mask = complaints_df["resolution_id"].str.strip() != ""
resolved_df = complaints_df[resolved_mask].copy()

print(f"Rules CSV ({rules_rel}): {len(rules)} rules loaded "
      f"(valid={sum(c == 'valid' for c, _ in rules)}, invalid={sum(c == 'invalid' for c, _ in rules)})")
print(f"Complaints total rows: {len(complaints_df):,}")
print(f"Resolved rows (resolution_id != ''): {len(resolved_df):,} "
      f"(unresolved empty resolution_id: {(~resolved_mask).sum():,})")

# Check ALL 152 lookup texts AND all distinct resolved texts in complaints_clean.csv for conflicts
lookup_conflicts = []
for rid, txt in zip(lookup_df["resolution_id"], lookup_df["text"]):
    try:
        classify_resolution_text(txt, return_rule=True)
    except ValueError as exc:
        lookup_conflicts.append((rid, str(exc)))

distinct_rids = sorted(resolved_df["resolution_id"].unique().tolist(), key=lambda x: int(x) if x.isdigit() else 999999)
lookup_text_map = dict(zip(lookup_df["resolution_id"], lookup_df["text"]))

resolved_conflicts = []
res_meta = {}
for rid in distinct_rids:
    txt = lookup_text_map[rid]
    try:
        cls, rule_str = classify_resolution_text(txt, return_rule=True)
    except ValueError as exc:
        resolved_conflicts.append((rid, str(exc)))
        cls, rule_str = "CONFLICT_ERROR", str(exc)
    res_meta[rid] = (txt, cls, rule_str)

print(f"Distinct resolved resolution_ids in complaints_clean.csv: {len(distinct_rids)}")
print(f"Conflicts raised across all {len(lookup_df)} lookup texts: {len(lookup_conflicts)}")
print(f"Conflicts raised across all {len(distinct_rids)} distinct resolved texts: {len(resolved_conflicts)}")

# Build Table (a): all 111 distinct resolved texts
grp_counts = resolved_df.groupby(["resolution_id", "group"]).size().unstack(fill_value=0)
tot_counts = resolved_df.groupby("resolution_id").size()
unclosed_counts = resolved_df.groupby("resolution_id")["unclosed"].sum()
empty_closed_counts = resolved_df.groupby("resolution_id")["empty_closed"].sum()

rows_a = []
for rid in distinct_rids:
    txt, cls, rule_str = res_meta[rid]
    norm_preview = " ".join(str(txt).split())[:90]
    row = {
        "id": rid,
        "text_90": norm_preview,
        "flood": int(grp_counts.loc[rid, "flood"]) if (rid in grp_counts.index and "flood" in grp_counts.columns) else 0,
        "noise": int(grp_counts.loc[rid, "noise"]) if (rid in grp_counts.index and "noise" in grp_counts.columns) else 0,
        "road": int(grp_counts.loc[rid, "road"]) if (rid in grp_counts.index and "road" in grp_counts.columns) else 0,
        "sewer_backup": int(grp_counts.loc[rid, "sewer_backup"]) if (rid in grp_counts.index and "sewer_backup" in grp_counts.columns) else 0,
        "total": int(tot_counts.get(rid, 0)),
        "empty_closed_cnt": int(empty_closed_counts.get(rid, 0)),
        "unclosed_cnt": int(unclosed_counts.get(rid, 0)),
        "class": cls,
        "matched_rule": rule_str if rule_str else "(no match)",
    }
    rows_a.append(row)

table_a_df = (
    pd.DataFrame(rows_a)
    .sort_values(["total", "id"], ascending=[False, True])
    .reset_index(drop=True)
)
table_a_df.insert(0, "rank", np.arange(1, len(table_a_df) + 1))

print(f"\n=== Table (a): All {len(table_a_df)} Distinct Resolved Texts (Sorted by Total Count Desc) ===")
print(table_a_df.to_string(index=False))

no_match_df = table_a_df[table_a_df["matched_rule"] == "(no match)"].copy()
print(f"\n=== Texts With No Rule Match ({len(no_match_df)} distinct texts, {no_match_df['total'].sum():,} resolved rows) ===")
print(no_match_df[["rank", "id", "total", "unclosed_cnt", "class", "text_90"]].to_string(index=False))


Rules CSV (src/models/trust_label_rules.csv): 37 rules loaded (valid=21, invalid=16)
Complaints total rows: 236,507
Resolved rows (resolution_id != ''): 233,399 (unresolved empty resolution_id: 3,108)


Distinct resolved resolution_ids in complaints_clean.csv: 111
Conflicts raised across all 152 lookup texts: 0
Conflicts raised across all 111 distinct resolved texts: 0



=== Table (a): All 111 Distinct Resolved Texts (Sorted by Total Count Desc) ===
 rank  id                                                                                    text_90  flood  noise  road  sewer_backup  total  empty_closed_cnt  unclosed_cnt    class                                                     matched_rule
    1   2 The New York City Police Department responded to the complaint and their investigation det      0  70415     0             0  70415                 0             1 excluded                                                       (no match)
    2   1 The New York City Police Department responded to the complaint and with the information av      0  65806     0             0  65806                 0             3  invalid                     observed no evidence of a criminal violation
    3   6 The New York City Police Department responded to the complaint and their investigation det      0  20286     0             0  20286                 0             0  

In [3]:
# Section (b): Pre-Gate vs Post-Gate Class Totals per Group & Gate vs Unclosed Explanation
complaints_df["label_pre_gate"] = assign_trust_labels(complaints_df, lookup_df=lookup_df, apply_gate=False)
complaints_df["label_post_gate"] = assign_trust_labels(complaints_df, lookup_df=lookup_df, apply_gate=True)

groups_order = ["flood", "noise", "road", "sewer_backup"]
cls_order = ["valid", "invalid", "excluded"]

def group_class_table(df_in: pd.DataFrame, label_col: str) -> pd.DataFrame:
    recs = []
    for g in groups_order + ["ALL"]:
        sub = df_in if g == "ALL" else df_in[df_in["group"] == g]
        vc = sub[label_col].value_counts()
        v = int(vc.get("valid", 0))
        iv = int(vc.get("invalid", 0))
        ex = int(vc.get("excluded", 0))
        recs.append({
            "group": g,
            "valid": v,
            "invalid": iv,
            "excluded": ex,
            "total": v + iv + ex,
        })
    return pd.DataFrame(recs)

pre_gate_resolved_tbl = group_class_table(complaints_df[resolved_mask], "label_pre_gate")
pre_gate_all_tbl = group_class_table(complaints_df, "label_pre_gate")
post_gate_all_tbl = group_class_table(complaints_df, "label_post_gate")

print("=== Table (b1): Pre-Gate Class Totals per Group (Resolved Rows Only, N=233,399) ===")
print(pre_gate_resolved_tbl.to_string(index=False))

print("\n=== Table (b2): Pre-Gate Class Totals per Group (All Rows incl. Empty resolution_id, N=236,507) ===")
print(pre_gate_all_tbl.to_string(index=False))

print("\n=== Table (b3): Post-Gate Class Totals per Group (All Rows, Gate excludes empty closed_date or resolution_id) ===")
print(post_gate_all_tbl.to_string(index=False))

# Explain 4,414 (unclosed) vs 4,175 (empty closed_date) in exact component counts
n_empty_closed_all = int(empty_closed_mask.sum())
n_status_not_closed_all = int(non_closed_status_mask.sum())
n_both_all = int((empty_closed_mask & non_closed_status_mask).sum())
n_only_empty_closed_all = int((empty_closed_mask & (~non_closed_status_mask)).sum())
n_only_status_not_closed_all = int(((~empty_closed_mask) & non_closed_status_mask).sum())
n_unclosed_union_all = int(unclosed_mask.sum())

n_empty_closed_res = int((empty_closed_mask & resolved_mask).sum())
n_status_not_closed_res = int((non_closed_status_mask & resolved_mask).sum())
n_both_res = int((empty_closed_mask & non_closed_status_mask & resolved_mask).sum())
n_only_status_not_closed_res = int(((~empty_closed_mask) & non_closed_status_mask & resolved_mask).sum())
n_unclosed_union_res = int((unclosed_mask & resolved_mask).sum())

gate_comp_df = pd.DataFrame([
    {"component": "1. empty closed_date (closed_date == '') [Step 6.1d Gate]", "all_rows": n_empty_closed_all, "resolved_rows": n_empty_closed_res, "unresolved_rows": n_empty_closed_all - n_empty_closed_res},
    {"component": "2. status != 'Closed'", "all_rows": n_status_not_closed_all, "resolved_rows": n_status_not_closed_res, "unresolved_rows": n_status_not_closed_all - n_status_not_closed_res},
    {"component": "3. Both (empty closed_date & status != 'Closed')", "all_rows": n_both_all, "resolved_rows": n_both_res, "unresolved_rows": n_both_all - n_both_res},
    {"component": "4. Only empty closed_date (closed_date == '' & status == 'Closed')", "all_rows": n_only_empty_closed_all, "resolved_rows": int((empty_closed_mask & (~non_closed_status_mask) & resolved_mask).sum()), "unresolved_rows": int((empty_closed_mask & (~non_closed_status_mask) & (~resolved_mask)).sum())},
    {"component": "5. Only status != 'Closed' (closed_date != '' & status != 'Closed')", "all_rows": n_only_status_not_closed_all, "resolved_rows": n_only_status_not_closed_res, "unresolved_rows": n_only_status_not_closed_all - n_only_status_not_closed_res},
    {"component": "6. Union unclosed = (closed_date == '') | (status != 'Closed')", "all_rows": n_unclosed_union_all, "resolved_rows": n_unclosed_union_res, "unresolved_rows": n_unclosed_union_all - n_unclosed_union_res},
    {"component": "7. Gate total excluded by closure/res gate = empty_closed | empty_res_id", "all_rows": int((empty_closed_mask | (~resolved_mask)).sum()), "resolved_rows": n_empty_closed_res, "unresolved_rows": int((~resolved_mask).sum())},
])
print("\n=== Table (b4): Gate vs Unclosed Component Decomposition (Explaining 4,414 vs 4,175) ===")
print(gate_comp_df.to_string(index=False))

extra_239 = complaints_df[(~empty_closed_mask) & non_closed_status_mask]
print("\nBreakdown of the 239 rows where closed_date != '' yet status != 'Closed':")
print("  By status:", extra_239["status"].value_counts().to_dict())
print("  By pre-gate class:", extra_239["label_pre_gate"].value_counts().to_dict())


=== Table (b1): Pre-Gate Class Totals per Group (Resolved Rows Only, N=233,399) ===
       group  valid  invalid  excluded  total
       flood   2445     1221      1602   5268
       noise   5873   107842     95365 209080
        road   5654     2324      8087  16065
sewer_backup    594     1381      1011   2986
         ALL  14566   112768    106065 233399

=== Table (b2): Pre-Gate Class Totals per Group (All Rows incl. Empty resolution_id, N=236,507) ===
       group  valid  invalid  excluded  total
       flood   2445     1221      1854   5520
       noise   5873   107842     98128 211843
        road   5654     2324      8160  16138
sewer_backup    594     1381      1031   3006
         ALL  14566   112768    109173 236507

=== Table (b3): Post-Gate Class Totals per Group (All Rows, Gate excludes empty closed_date or resolution_id) ===
       group  valid  invalid  excluded  total
       flood   2442     1221      1857   5520
       noise   5871   107830     98142 211843
        ro

In [4]:
# Section (c) & (d): S1/S2/S3 Sensitivity Totals & Per-Group Invalid Rates
sens_configs = [
    ("Base", None),
    ("S1 ('no criminal violation existed' -> invalid)", "S1"),
    ("S2 ('no criminal violation existed' -> valid)", "S2"),
    ("S3 ('unable to gain entry' -> invalid)", "S3"),
]

sens_rows_pre = []
sens_rows_post = []
sens_labels_cache = {}
for s_name, s_arg in sens_configs:
    lbl_pre = assign_trust_labels(complaints_df, lookup_df=lookup_df, apply_gate=False, sensitivity=s_arg)
    lbl_post = assign_trust_labels(complaints_df, lookup_df=lookup_df, apply_gate=True, sensitivity=s_arg)
    sens_labels_cache[s_arg or "BASE"] = (lbl_pre, lbl_post)

    vc_pre_res = lbl_pre[resolved_mask].value_counts()
    sens_rows_pre.append({
        "sensitivity": s_name,
        "valid": int(vc_pre_res.get("valid", 0)),
        "invalid": int(vc_pre_res.get("invalid", 0)),
        "excluded": int(vc_pre_res.get("excluded", 0)),
        "resolved_total": int(resolved_mask.sum()),
    })

    vc_post_all = lbl_post.value_counts()
    sens_rows_post.append({
        "sensitivity": s_name,
        "valid": int(vc_post_all.get("valid", 0)),
        "invalid": int(vc_post_all.get("invalid", 0)),
        "excluded": int(vc_post_all.get("excluded", 0)),
        "all_total": len(complaints_df),
    })

sens_pre_df = pd.DataFrame(sens_rows_pre)
sens_post_df = pd.DataFrame(sens_rows_post)

print("=== Table (c1): Sensitivity Class Totals — Pre-Gate (Resolved Rows, N=233,399) ===")
print(sens_pre_df.to_string(index=False))

print("\n=== Table (c2): Sensitivity Class Totals — Post-Gate (All Rows, N=236,507) ===")
print(sens_post_df.to_string(index=False))

# Table (d): Per-group invalid rate over valid + invalid, overall and noise-only
invalid_rate_rows = []
for slice_label, mask_s in [
    ("flood", complaints_df["group"] == "flood"),
    ("noise", complaints_df["group"] == "noise"),
    ("road", complaints_df["group"] == "road"),
    ("sewer_backup", complaints_df["group"] == "sewer_backup"),
    ("NON-NOISE (flood+road+sewer)", complaints_df["group"] != "noise"),
    ("OVERALL (all 4 groups)", np.ones(len(complaints_df), dtype=bool)),
]:
    sub_post = complaints_df.loc[mask_s, "label_post_gate"].value_counts()
    v_po = int(sub_post.get("valid", 0))
    iv_po = int(sub_post.get("invalid", 0))
    den_po = v_po + iv_po
    inv_rate_po = (iv_po / den_po * 100.0) if den_po > 0 else float("nan")

    sub_pre = complaints_df.loc[mask_s & resolved_mask, "label_pre_gate"].value_counts()
    v_pr = int(sub_pre.get("valid", 0))
    iv_pr = int(sub_pre.get("invalid", 0))
    den_pr = v_pr + iv_pr
    inv_rate_pr = (iv_pr / den_pr * 100.0) if den_pr > 0 else float("nan")

    invalid_rate_rows.append({
        "scope": slice_label,
        "post_valid": v_po,
        "post_invalid": iv_po,
        "post_labelled": den_po,
        "post_invalid_rate_pct": round(inv_rate_po, 2),
        "pre_valid": v_pr,
        "pre_invalid": iv_pr,
        "pre_labelled": den_pr,
        "pre_invalid_rate_pct": round(inv_rate_pr, 2),
    })

table_d_df = pd.DataFrame(invalid_rate_rows)
print("\n=== Table (d): Invalid Rate Over (Valid + Invalid) — Per Group, Noise-Only, and Overall ===")
print(table_d_df.to_string(index=False, formatters={
    "post_invalid_rate_pct": "{:.2f}%".format,
    "pre_invalid_rate_pct": "{:.2f}%".format,
}))


=== Table (c1): Sensitivity Class Totals — Pre-Gate (Resolved Rows, N=233,399) ===
                                    sensitivity  valid  invalid  excluded  resolved_total
                                           Base  14566   112768    106065          233399
S1 ('no criminal violation existed' -> invalid)  14566   183183     35650          233399
  S2 ('no criminal violation existed' -> valid)  84981   112768     35650          233399
         S3 ('unable to gain entry' -> invalid)  14566   128229     90604          233399

=== Table (c2): Sensitivity Class Totals — Post-Gate (All Rows, N=236,507) ===
                                    sensitivity  valid  invalid  excluded  all_total
                                           Base  14558   112755    109194     236507
S1 ('no criminal violation existed' -> invalid)  14558   183170     38779     236507
  S2 ('no criminal violation existed' -> valid)  84973   112755     38779     236507
         S3 ('unable to gain entry' -> invalid)


=== Table (d): Invalid Rate Over (Valid + Invalid) — Per Group, Noise-Only, and Overall ===
                       scope  post_valid  post_invalid  post_labelled post_invalid_rate_pct  pre_valid  pre_invalid  pre_labelled pre_invalid_rate_pct
                       flood        2442          1221           3663                33.33%       2445         1221          3666               33.31%
                       noise        5871        107830         113701                94.84%       5873       107842        113715               94.84%
                        road        5652          2324           7976                29.14%       5654         2324          7978               29.13%
                sewer_backup         593          1380           1973                69.94%        594         1381          1975               69.92%
NON-NOISE (flood+road+sewer)        8687          4925          13612                36.18%       8693         4926         13619               36.17%
 

In [5]:
# Section (e), (f), (g): Chronological Split Balance, Road Monthly Drift (#8 & #20), and Noise Labelled Share
# (e) Existing chronological 80/20 split by timestamp
complaints_sorted = complaints_df.sort_values("ts").reset_index(drop=True)
boundary_ts = complaints_sorted.loc[int(len(complaints_sorted) * 0.8), "ts"]
train_mask = complaints_df["ts"] < boundary_ts
test_mask = complaints_df["ts"] >= boundary_ts

print(f"Chronological 80/20 boundary: {boundary_ts} | Train: {int(train_mask.sum()):,} | Test: {int(test_mask.sum()):,}")

split_rows = []
for g in groups_order + ["ALL"]:
    g_mask = np.ones(len(complaints_df), dtype=bool) if g == "ALL" else (complaints_df["group"] == g).to_numpy()
    tr_vc = complaints_df.loc[train_mask & g_mask, "label_post_gate"].value_counts()
    te_vc = complaints_df.loc[test_mask & g_mask, "label_post_gate"].value_counts()
    tr_v, tr_i, tr_e = int(tr_vc.get("valid", 0)), int(tr_vc.get("invalid", 0)), int(tr_vc.get("excluded", 0))
    te_v, te_i, te_e = int(te_vc.get("valid", 0)), int(te_vc.get("invalid", 0)), int(te_vc.get("excluded", 0))
    split_rows.append({
        "group": g,
        "train_valid": tr_v,
        "train_invalid": tr_i,
        "train_excluded": tr_e,
        "train_total": tr_v + tr_i + tr_e,
        "test_valid": te_v,
        "test_invalid": te_i,
        "test_excluded": te_e,
        "test_total": te_v + te_i + te_e,
    })

table_e_df = pd.DataFrame(split_rows)
print("\n=== Table (e): Train / Test Post-Gate Class Counts per Group (Boundary 2026-09-01 00:16:13) ===")
print(table_e_df.to_string(index=False))

# (f) Road: created month x class table + texts #8 (ID 25, DOT->DEP 60d) & #20 (ID 21, DOT->Maint) + unclosed share
road_df = complaints_df[complaints_df["group"] == "road"].copy()
rank8_id = str(table_a_df.loc[table_a_df["rank"] == 8, "id"].iloc[0])
rank20_id = str(table_a_df.loc[table_a_df["rank"] == 20, "id"].iloc[0])
rank8_txt = lookup_text_map[rank8_id]
rank20_txt = lookup_text_map[rank20_id]

print(f"\nRoad target resolution texts verified by overall resolved frequency rank:")
print(f"  Rank #8  -> resolution_id='{rank8_id}': {rank8_txt}")
print(f"  Rank #20 -> resolution_id='{rank20_id}': {rank20_txt}")

road_months = sorted(road_df["month_created"].unique())
road_month_rows = []
road_texts_rows = []
for m in road_months:
    rm = road_df[road_df["month_created"] == m]
    vc = rm["label_post_gate"].value_counts()
    v, iv, ex = int(vc.get("valid", 0)), int(vc.get("invalid", 0)), int(vc.get("excluded", 0))
    tot = len(rm)
    unc_n = int(rm["unclosed"].sum())
    emp_n = int(rm["empty_closed"].sum())
    road_month_rows.append({
        "month": m,
        "valid": v,
        "invalid": iv,
        "excluded": ex,
        "total": tot,
        "valid_pct": (v / tot * 100.0) if tot else 0.0,
        "excluded_pct": (ex / tot * 100.0) if tot else 0.0,
        "empty_closed_n": emp_n,
        "empty_closed_pct": (emp_n / tot * 100.0) if tot else 0.0,
        "unclosed_n": unc_n,
        "unclosed_pct": (unc_n / tot * 100.0) if tot else 0.0,
    })

    c8 = int((rm["resolution_id"] == rank8_id).sum())
    c20 = int((rm["resolution_id"] == rank20_id).sum())
    road_texts_rows.append({
        "month": m,
        f"text8_id{rank8_id}_DOT_DEP_60d": c8,
        f"text20_id{rank20_id}_DOT_Maint": c20,
        "combined_8_20": c8 + c20,
        "road_total": tot,
        "combined_share_pct": ((c8 + c20) / tot * 100.0) if tot else 0.0,
        "unclosed_n": unc_n,
        "unclosed_share_pct": (unc_n / tot * 100.0) if tot else 0.0,
    })

# Append TOTAL row for road tables
road_tot = len(road_df)
road_v = int((road_df["label_post_gate"] == "valid").sum())
road_iv = int((road_df["label_post_gate"] == "invalid").sum())
road_ex = int((road_df["label_post_gate"] == "excluded").sum())
road_unc = int(road_df["unclosed"].sum())
road_emp = int(road_df["empty_closed"].sum())
road_month_rows.append({
    "month": "ALL",
    "valid": road_v,
    "invalid": road_iv,
    "excluded": road_ex,
    "total": road_tot,
    "valid_pct": road_v / road_tot * 100.0,
    "excluded_pct": road_ex / road_tot * 100.0,
    "empty_closed_n": road_emp,
    "empty_closed_pct": road_emp / road_tot * 100.0,
    "unclosed_n": road_unc,
    "unclosed_pct": road_unc / road_tot * 100.0,
})
c8_all = int((road_df["resolution_id"] == rank8_id).sum())
c20_all = int((road_df["resolution_id"] == rank20_id).sum())
road_texts_rows.append({
    "month": "ALL",
    f"text8_id{rank8_id}_DOT_DEP_60d": c8_all,
    f"text20_id{rank20_id}_DOT_Maint": c20_all,
    "combined_8_20": c8_all + c20_all,
    "road_total": road_tot,
    "combined_share_pct": (c8_all + c20_all) / road_tot * 100.0,
    "unclosed_n": road_unc,
    "unclosed_share_pct": road_unc / road_tot * 100.0,
})

table_f1_df = pd.DataFrame(road_month_rows)
table_f2_df = pd.DataFrame(road_texts_rows)

print("\n=== Table (f1): Road Created Month x Post-Gate Class & Unclosed Share ===")
print(table_f1_df.to_string(index=False, formatters={
    "valid_pct": "{:.2f}%".format,
    "excluded_pct": "{:.2f}%".format,
    "empty_closed_pct": "{:.2f}%".format,
    "unclosed_pct": "{:.2f}%".format,
}))

print(f"\n=== Table (f2): Road Monthly Counts for Text #8 (ID {rank8_id}) & Text #20 (ID {rank20_id}) + Unclosed Share ===")
print(table_f2_df.to_string(index=False, formatters={
    "combined_share_pct": "{:.2f}%".format,
    "unclosed_share_pct": "{:.2f}%".format,
}))

# (g) Noise labelled share of rows: train vs test
noise_share_rows = []
for split_lbl, m_sp in [
    ("Train (< 2026-09-01 00:16:13)", train_mask & (complaints_df["group"] == "noise")),
    ("Test (>= 2026-09-01 00:16:13)", test_mask & (complaints_df["group"] == "noise")),
    ("Overall (Noise All)", (complaints_df["group"] == "noise")),
]:
    sub = complaints_df[m_sp]
    vc = sub["label_post_gate"].value_counts()
    v, iv, ex = int(vc.get("valid", 0)), int(vc.get("invalid", 0)), int(vc.get("excluded", 0))
    tot = len(sub)
    lab = v + iv
    noise_share_rows.append({
        "partition": split_lbl,
        "total_rows": tot,
        "valid": v,
        "invalid": iv,
        "excluded": ex,
        "labelled_rows": lab,
        "labelled_share_pct": (lab / tot * 100.0) if tot else 0.0,
        "invalid_of_labelled_pct": (iv / lab * 100.0) if lab else 0.0,
    })

table_g_df = pd.DataFrame(noise_share_rows)
print("\n=== Table (g): Noise Labelled Share ((valid + invalid) / total) — Train vs Test ===")
print(table_g_df.to_string(index=False, formatters={
    "labelled_share_pct": "{:.2f}%".format,
    "invalid_of_labelled_pct": "{:.2f}%".format,
}))


Chronological 80/20 boundary: 2026-09-01 00:16:13 | Train: 189,205 | Test: 47,302



=== Table (e): Train / Test Post-Gate Class Counts per Group (Boundary 2026-09-01 00:16:13) ===
       group  train_valid  train_invalid  train_excluded  train_total  test_valid  test_invalid  test_excluded  test_total
       flood         2124           1085            1503         4712         318           136            354         808
       noise         5191          85857           77208       168256         680         21973          20934       43587
        road         4934           1993            6731        13658         718           331           1431        2480
sewer_backup          554           1162             863         2579          39           218            170         427
         ALL        12803          90097           86305       189205        1755         22658          22889       47302

Road target resolution texts verified by overall resolved frequency rank:
  Rank #8  -> resolution_id='25': The Department of Transportation inspected and has reque


=== Table (f1): Road Created Month x Post-Gate Class & Unclosed Share ===
  month  valid  invalid  excluded  total valid_pct excluded_pct  empty_closed_n empty_closed_pct  unclosed_n unclosed_pct
2025-11    224       80       153    457    49.02%       33.48%               0            0.00%           0        0.00%
2025-12    399       81       162    642    62.15%       25.23%               2            0.31%           2        0.31%
2026-01    281       60       200    541    51.94%       36.97%              22            4.07%          22        4.07%
2026-02    378       80       281    739    51.15%       38.02%              27            3.65%          27        3.65%
2026-03   1119      338      1081   2538    44.09%       42.59%             119            4.69%         119        4.69%
2026-04    736      226       871   1833    40.15%       47.52%              84            4.58%          84        4.58%
2026-05    459      241       849   1549    29.63%       54.81%        


=== Table (g): Noise Labelled Share ((valid + invalid) / total) — Train vs Test ===
                    partition  total_rows  valid  invalid  excluded  labelled_rows labelled_share_pct invalid_of_labelled_pct
Train (< 2026-09-01 00:16:13)      168256   5191    85857     77208          91048             54.11%                  94.30%
Test (>= 2026-09-01 00:16:13)       43587    680    21973     20934          22653             51.97%                  97.00%
          Overall (Noise All)      211843   5871   107830     98142         113701             53.67%                  94.84%


In [6]:
# Section Checks: Step 6.1d Verified Totals & Protected File Hashes
chk = Checker()

# 1. Core counts
n_rows = len(complaints_df)
n_distinct_texts = len(distinct_rids)
n_resolved_rows = int(resolved_mask.sum())
n_conflicts_all = len(lookup_conflicts) + len(resolved_conflicts)
n_resolved_no_closed = n_empty_closed_res

chk.check("Total rows == 236507", n_rows, 236507)
chk.check("Distinct resolved texts == 111", n_distinct_texts, 111)
chk.check("Resolved rows == 233399", n_resolved_rows, 233399)
chk.check("Texts with no rule match listed (>0) & conflicts raised == 0", (len(no_match_df) > 0) and (n_conflicts_all == 0), True)
chk.check("Resolved rows without closed_date == 1068", n_resolved_no_closed, 1068)

# 2. Post-gate class totals (all 236,507 rows)
pg_all = post_gate_all_tbl.set_index("group").loc["ALL"]
chk.check("Post-gate valid == 14558", int(pg_all["valid"]), 14558)
chk.check("Post-gate invalid == 112755", int(pg_all["invalid"]), 112755)
chk.check("Post-gate excluded == 109194", int(pg_all["excluded"]), 109194)
chk.check("Post-gate sum == 236507", int(pg_all["valid"] + pg_all["invalid"] + pg_all["excluded"]), 236507)

# 3. Pre-gate class totals (resolved 233,399 rows) & noise pre-gate
pr_all = pre_gate_resolved_tbl.set_index("group").loc["ALL"]
pr_noise = pre_gate_resolved_tbl.set_index("group").loc["noise"]
chk.check("Pre-gate resolved valid == 14566", int(pr_all["valid"]), 14566)
chk.check("Pre-gate resolved invalid == 112768", int(pr_all["invalid"]), 112768)
chk.check("Pre-gate resolved excluded == 106065", int(pr_all["excluded"]), 106065)
chk.check("Noise pre-gate valid == 5873", int(pr_noise["valid"]), 5873)
chk.check("Noise pre-gate invalid == 107842", int(pr_noise["invalid"]), 107842)

# 4. Sensitivities S1 / S2 / S3 (pre-gate resolved rows)
s_pre_map = sens_pre_df.set_index("sensitivity")
s1_row = s_pre_map.loc[[i for i in s_pre_map.index if i.startswith("S1")][0]]
s2_row = s_pre_map.loc[[i for i in s_pre_map.index if i.startswith("S2")][0]]
s3_row = s_pre_map.loc[[i for i in s_pre_map.index if i.startswith("S3")][0]]
chk.check("S1 pre-gate invalid == 183183", int(s1_row["invalid"]), 183183)
chk.check("S2 pre-gate valid == 84981", int(s2_row["valid"]), 84981)
chk.check("S3 pre-gate invalid == 128229", int(s3_row["invalid"]), 128229)

# 5. Split & road rank checks
chk.check("Chrono split boundary == 2026-09-01 00:16:13", str(boundary_ts), "2026-09-01 00:16:13")
chk.check("Train / test sizes == 189205 / 47302", (int(train_mask.sum()), int(test_mask.sum())), (189205, 47302))
chk.check("Rank #8 is ID 25 (DOT->DEP 60d) & Rank #20 is ID 21 (DOT->Maint)", (rank8_id, rank20_id), ("25", "21"))

# 6. Protected file SHA-256 verification
h_cc = hashlib.sha256((ROOT / "data/processed/complaints_clean.csv").read_bytes()).hexdigest()
chk.check("complaints_clean.csv SHA256 unchanged", h_cc, "fb0fa8f176b04c8258efa490d63855c572bc055f2272e9d138a3c0532b50a4d7")
h_inc = hashlib.sha256((ROOT / "data/processed/incidents.csv").read_bytes()).hexdigest()
chk.check("incidents.csv SHA256 unchanged", h_inc, "fe2eea51a8af06aa372930286e2aa64bfa015346d87837ed47543b3594e1583e")
h_map = hashlib.sha256((ROOT / "data/processed/complaint_incident_map.csv").read_bytes()).hexdigest()
chk.check("complaint_incident_map.csv SHA256 unchanged", h_map, "3fbf170ba82b5e187ef24d69c9bec0715cd1ac6a501bb99e2521713b65503f19")
h_fm = hashlib.sha256((ROOT / "models/flood_model.joblib").read_bytes()).hexdigest().upper()
chk.check("models/flood_model.joblib SHA256 unchanged", h_fm, "7BD63D00B54A7578A1C1B41DA28A64C98DDEAC6831A4922EC209DF78FA94B2CE")
h_nm = hashlib.sha256((ROOT / "models/noise_model.joblib").read_bytes()).hexdigest().upper()
chk.check("models/noise_model.joblib SHA256 unchanged", h_nm, "39D0E6B3E6B1B7CBE0C68578F1719B307350610C9CA96987C69EB7460229F201")

# Diagnostic printout if any check failed
if chk.failed > 0:
    print("\nFAIL DIAGNOSTIC: Per-text counts & assigned classes:")
    print(table_a_df[["rank", "id", "total", "unclosed_cnt", "class", "matched_rule", "text_90"]].to_string(index=False))

chk.summary()


[PASS] Total rows == 236507: actual=236507, expected=236507
[PASS] Distinct resolved texts == 111: actual=111, expected=111
[PASS] Resolved rows == 233399: actual=233399, expected=233399
[PASS] Texts with no rule match listed (>0) & conflicts raised == 0: actual=True, expected=True
[PASS] Resolved rows without closed_date == 1068: actual=1068, expected=1068
[PASS] Post-gate valid == 14558: actual=14558, expected=14558
[PASS] Post-gate invalid == 112755: actual=112755, expected=112755
[PASS] Post-gate excluded == 109194: actual=109194, expected=109194
[PASS] Post-gate sum == 236507: actual=236507, expected=236507
[PASS] Pre-gate resolved valid == 14566: actual=14566, expected=14566
[PASS] Pre-gate resolved invalid == 112768: actual=112768, expected=112768
[PASS] Pre-gate resolved excluded == 106065: actual=106065, expected=106065
[PASS] Noise pre-gate valid == 5873: actual=5873, expected=5873
[PASS] Noise pre-gate invalid == 107842: actual=107842, expected=107842
[PASS] S1 pre-gate inva

[PASS] complaints_clean.csv SHA256 unchanged: actual=fb0fa8f176b04c8258efa490d63855c572bc055f2272e9d138a3c0532b50a4d7, expected=fb0fa8f176b04c8258efa490d63855c572bc055f2272e9d138a3c0532b50a4d7
[PASS] incidents.csv SHA256 unchanged: actual=fe2eea51a8af06aa372930286e2aa64bfa015346d87837ed47543b3594e1583e, expected=fe2eea51a8af06aa372930286e2aa64bfa015346d87837ed47543b3594e1583e


[PASS] complaint_incident_map.csv SHA256 unchanged: actual=3fbf170ba82b5e187ef24d69c9bec0715cd1ac6a501bb99e2521713b65503f19, expected=3fbf170ba82b5e187ef24d69c9bec0715cd1ac6a501bb99e2521713b65503f19
[PASS] models/flood_model.joblib SHA256 unchanged: actual=7BD63D00B54A7578A1C1B41DA28A64C98DDEAC6831A4922EC209DF78FA94B2CE, expected=7BD63D00B54A7578A1C1B41DA28A64C98DDEAC6831A4922EC209DF78FA94B2CE
[PASS] models/noise_model.joblib SHA256 unchanged: actual=39D0E6B3E6B1B7CBE0C68578F1719B307350610C9CA96987C69EB7460229F201, expected=39D0E6B3E6B1B7CBE0C68578F1719B307350610C9CA96987C69EB7460229F201
CHECKS: 25 passed, 0 failed


## Key Findings & Modeling Context

1. **Standardized Single-Rule CSV Architecture (`src/models/trust_label_rules.csv`)**:
   - All 37 CSV rules (`21` valid, `16` invalid) evaluate cleanly with **0 conflicts** across all 152 lookup entries and all 111 distinct resolved texts in `complaints_clean.csv`; **60** distinct resolved texts match no CSV rule and default to `excluded`.
   - Across all 236,507 complaints, post-gate totals partition into **14,558** `valid` (6.16%), **112,755** `invalid` (47.68%), and **109,194** `excluded` (46.17%).

2. **Gate (`empty closed_date`) vs `unclosed` Decomposition**:
   - `empty_closed_date` (`closed_date == ''`) affects **4,175** rows (`1,068` resolved + `3,107` unresolved), whereas `status != 'Closed'` affects **4,312** rows (`4,073` overlapping `empty_closed_date` + `239` resolved rows with a non-empty `closed_date` but `'In Progress'`/`'Unspecified'` status), giving union `unclosed = 4,414`.

3. **Sensitivities `S1`–`S3` & Subgroup Invalid Rates**:
   - Under pre-gate resolved totals, `S1` increases `invalid` from **112,768** to **183,183** (+70,415 rows from NYPD `no criminal violation existed`), `S2` lifts `valid` from **14,566** to **84,981**, and `S3` raises `invalid` to **128,229** (+15,461 `unable to gain entry` rows).
   - Post-gate invalid rate over `valid + invalid` is **88.57%** overall (`112,755 / 127,313`), driven primarily by **94.84%** in `noise` (`107,830 / 113,701`), versus **69.94%** in `sewer_backup`, **33.33%** in `flood`, and **29.14%** in `road`.

4. **Road Temporal Drift & Noise Labelled Share**:
   - Under the chronological 80/20 split (`2026-09-01 00:16:13`, 189,205 train / 47,302 test), `road` exhibits an operational shift toward referrals and unclosed records in late 2026 (including text #8 `id=25` DOT->DEP 60-day inspections and text #20 `id=21` DOT Maintenance referrals).
   - `noise` labelled share (`(valid + invalid) / total`) stays stable across the split: **54.11%** in train (`91,048 / 168,256`) vs **51.97%** in test (`22,653 / 43,587`).

## Modeling Limitations

1. **Large Excluded Boilerplate Pool**:
   - Nearly half of all complaints (`46.17%` post-gate) map to `excluded`, dominated by `resolution_id='2'` (`70,415` rows stating `no criminal violation existed; corrected without summons/arrest`), creating sensitivity to whether those closures represent valid or invalid citizen complaints (`S1` vs `S2`).

2. **Extreme Cross-Domain Class Imbalance**:
   - Because `noise` represents `89.31%` of all labelled rows (`113,701 / 127,313`) with a `94.84%` invalid rate, global metrics and unstratified classifiers are heavily weighted by NYPD noise dispositions.

3. **Late-Period Road Censoring & Referral Shift**:
   - Long-latency DOT referrals (`id=25`, `id=21`) and unclosed tickets cluster heavily in September–October 2026, reducing the labelled fraction in the final test months for `road`.